# Graph Export & Cost Analysis

This notebook shows how to export and visualize the knowledge graph, inspect structured query output, and analyze the cost profile of Neocortex operations.

You'll learn:
1. Export the graph to GraphML format
2. Visualize it with NetworkX and matplotlib
3. Access structured response data via `to_dict()`
4. Track and chart cost/latency across queries

In [ ]:
import os
import time
import json
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

from neocortex import GraphRAG, QueryParam
from neocortex._llm import token_tracker

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

# Optional visualization dependencies
try:
    import networkx as nx
    import matplotlib.pyplot as plt
    HAS_VIZ = True
    print("Visualization libraries available (networkx + matplotlib).")
except ImportError:
    HAS_VIZ = False
    print("networkx or matplotlib not installed. Install with:")
    print("  pip install networkx matplotlib")
    print("Graph visualization cells will be skipped.")

print("Environment ready.")

## Build a Small Knowledge Graph

We'll index a compact, entity-rich text to get a nicely connected graph for visualization.

In [ ]:
SAMPLE_TEXT = """
The Manhattan Project was a research and development program during World War II that
produced the first nuclear weapons. It was led by the United States with support from
the United Kingdom and Canada. The project was directed by Major General Leslie Groves
of the U.S. Army Corps of Engineers. Physicist J. Robert Oppenheimer served as the
director of the Los Alamos Laboratory in New Mexico, where the weapons were designed.

The project employed more than 125,000 workers and cost nearly $2 billion (equivalent
to about $28 billion in 2024 dollars). Key research sites included Oak Ridge in
Tennessee, which produced enriched uranium; Hanford in Washington state, which
manufactured plutonium; and Los Alamos, where the bomb designs were developed.

The first nuclear test, codenamed "Trinity," took place on July 16, 1945, at the
Alamogordo Bombing Range in New Mexico. The test was a success and demonstrated the
implosion-type nuclear weapon design. Oppenheimer later recalled that the explosion
brought to mind a line from the Hindu scripture Bhagavad Gita: "Now I am become Death,
the destroyer of worlds."

Enrico Fermi, an Italian physicist, led the team that achieved the first self-sustaining
nuclear chain reaction on December 2, 1942, at the University of Chicago. This
experiment, known as Chicago Pile-1, was a crucial milestone for the Manhattan Project.
Niels Bohr, the Danish physicist and Nobel laureate, also contributed as a consultant.
Leo Szilard, a Hungarian-American physicist, had originally conceived the nuclear chain
reaction and co-authored a letter with Albert Einstein to President Franklin Roosevelt
in 1939, warning of the potential for nuclear weapons and urging U.S. research.
""".strip()

rag = GraphRAG(
    working_dir="./db/examples_viz",
    domain="historical events, scientific research, and military projects related to nuclear physics",
    example_queries=(
        "Who led the Manhattan Project?\n"
        "Where was the first nuclear test?\n"
        "What was Enrico Fermi's contribution?"
    ),
    entity_types=["person", "organization", "location", "event", "project"],
)

token_tracker.reset()
start = time.perf_counter()

num_entities, num_relations, num_chunks = rag.insert(SAMPLE_TEXT)

elapsed = time.perf_counter() - start
index_cost = token_tracker.total_cost

print(f"Indexed in {elapsed:.1f}s (${index_cost:.4f})")
print(f"  Entities:  {num_entities}")
print(f"  Relations: {num_relations}")
print(f"  Chunks:    {num_chunks}")

## Export to GraphML

GraphML is an XML-based format supported by many graph analysis tools (Gephi, Cytoscape, NetworkX, etc.).

In [ ]:
GRAPHML_PATH = "./db/examples_viz/knowledge_graph.graphml"

rag.save_graphml(GRAPHML_PATH)

file_size = Path(GRAPHML_PATH).stat().st_size
print(f"GraphML exported to: {GRAPHML_PATH}")
print(f"File size: {file_size:,} bytes ({file_size / 1024:.1f} KB)")

## Visualize with NetworkX

Load the GraphML file and render the knowledge graph with nodes colored by entity type.

In [ ]:
if HAS_VIZ:
    G = nx.read_graphml(GRAPHML_PATH)

    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # Color nodes by type
    type_colors = {
        "PERSON": "#4A90D9",
        "ORGANIZATION": "#E07B53",
        "LOCATION": "#50C878",
        "EVENT": "#F5C542",
        "PROJECT": "#9B59B6",
    }
    default_color = "#CCCCCC"

    node_colors = []
    for node in G.nodes():
        node_type = G.nodes[node].get("type", "").upper()
        node_colors.append(type_colors.get(node_type, default_color))

    # Draw
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    pos = nx.spring_layout(G, k=2.0, seed=42)

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, font_weight="bold", ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.3, arrows=True, arrowsize=15, ax=ax)

    # Legend
    for label, color in type_colors.items():
        ax.scatter([], [], c=color, s=100, label=label.title())
    ax.legend(loc="upper left", framealpha=0.9)
    ax.set_title("Neocortex Knowledge Graph", fontsize=14)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping visualization (networkx/matplotlib not installed).")

## Structured Output

The `to_dict()` method converts a query response into a JSON-serializable dictionary — useful for logging, downstream processing, or API responses.

In [ ]:
response = rag.query(
    "Who were the key scientists in the Manhattan Project?",
    params=QueryParam.balanced(),
)

output = response.to_dict()
print(json.dumps(output, indent=2, default=str)[:3000])
if len(json.dumps(output, default=str)) > 3000:
    print("\n... (truncated)")

## Cost Analysis

Let's run multiple queries and collect detailed cost and latency data.

In [ ]:
QUESTIONS = [
    "Who led the Manhattan Project?",
    "What was the Trinity test?",
    "Describe Enrico Fermi's contribution to nuclear physics.",
    "What role did Albert Einstein play in the development of nuclear weapons?",
    "Where were the main research sites of the Manhattan Project?",
]

metrics = []

for question in QUESTIONS:
    token_tracker.reset()
    start = time.perf_counter()

    response = rag.query(question, params=QueryParam.balanced())

    elapsed = time.perf_counter() - start

    metrics.append({
        "question": question,
        "latency": elapsed,
        "total_cost": token_tracker.total_cost,
        "llm_cost": token_tracker.llm_cost,
        "embedding_cost": token_tracker.embedding_cost,
        "llm_input_tokens": token_tracker.llm_input_tokens,
        "llm_output_tokens": token_tracker.llm_output_tokens,
        "embedding_tokens": token_tracker.embedding_input_tokens,
    })

# Print summary table
print(f"{'#':<3} {'Latency':>8} {'Cost':>8} {'LLM In':>8} {'LLM Out':>8} {'Embed':>8}  Question")
print("-" * 95)
for i, m in enumerate(metrics):
    q_short = m['question'][:40]
    print(f"{i+1:<3} {m['latency']:>7.1f}s ${m['total_cost']:>6.4f} {m['llm_input_tokens']:>8} "
          f"{m['llm_output_tokens']:>8} {m['embedding_tokens']:>8}  {q_short}")

In [ ]:
if HAS_VIZ:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    labels = [f"Q{i+1}" for i in range(len(metrics))]
    colors = ["#4A90D9", "#E07B53", "#50C878", "#F5C542", "#9B59B6"]

    # Latency per query
    axes[0].bar(labels, [m["latency"] for m in metrics], color=colors)
    axes[0].set_title("Latency per Query")
    axes[0].set_ylabel("Seconds")

    # Cost per query
    axes[1].bar(labels, [m["total_cost"] for m in metrics], color=colors)
    axes[1].set_title("Cost per Query")
    axes[1].set_ylabel("USD")
    axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:.4f}"))

    # Token distribution (stacked bar)
    llm_in = [m["llm_input_tokens"] for m in metrics]
    llm_out = [m["llm_output_tokens"] for m in metrics]
    embed = [m["embedding_tokens"] for m in metrics]

    axes[2].bar(labels, llm_in, label="LLM Input", color="#4A90D9")
    axes[2].bar(labels, llm_out, bottom=llm_in, label="LLM Output", color="#E07B53")
    axes[2].bar(labels, embed, bottom=[a + b for a, b in zip(llm_in, llm_out)], label="Embedding", color="#50C878")
    axes[2].set_title("Token Distribution")
    axes[2].set_ylabel("Tokens")
    axes[2].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Skipping charts (matplotlib not installed).")

## Cost Model Summary

Neocortex costs come from two sources:

| Operation | LLM Calls | Embedding Calls | When |
|-----------|-----------|-----------------|------|
| **Indexing** (`insert`) | Entity/relation extraction | Chunk + entity + relation embeddings | Once per document |
| **Querying** (`query`) | Entity extraction + answer generation | Query embedding for similarity search | Per query |

**Cost reduction strategies:**
- Use `QueryParam.economy()` for batch or low-priority queries
- Use `only_context=True` to skip answer generation when you only need retrieval
- Index once, query many times — indexing is the expensive part
- Use `gpt-4o-mini` (the default) for a good balance of quality and cost